# Steerable Chatbots

Replicates the preference-based activation steering from **"Steerable Chatbots: Personalizing LLMs with Preference-Based Activation Steering"** ([arXiv:2505.04260](https://arxiv.org/abs/2505.04260v2)) on Qwen2.5-1.5B-Instruct, end to end in one engine:

1. **Construction** — hidden states are captured over adult-oriented vs child-oriented activity recommendations (`style_examples.json`), and a linear-probe control vector is fitted (`style-probe.gguf`).
2. **Steering** — a negative scale personalizes replies toward kid-friendly suggestions and a positive scale toward adults-only ones; both are shown against a baseline.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")  # Qwen/Qwen2.5-1.5B-Instruct

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)

## Vector construction

In [ ]:
import json

with open("style_examples.json", encoding="utf-8") as f:
    examples = json.load(f)

adult = examples["adult"]
child = examples["child"]

In [ ]:
from easysteer.capture import capture
from vllm.capture import SelectSpec

# Only the last prompt row feeds the extractor, so select it at the source.
result = capture(
    llm,
    adult + child,
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Fit the probe to adult (positive) versus child (negative) examples.
labels = [True] * len(adult) + [False] * len(child)
control_vector = extract(
    result,
    labels,
    method="linear_probe",
    token_pos=-1,
    normalize=True,
)
control_vector.export_gguf("style-probe.gguf")


## Steering

In [ ]:
# Baseline: family-activity request, no steering.
example = "<|im_start|>user\nWhat are some activities our family can do in the city this weekend?<|im_end|>\n<|im_start|>assistant\n"
params = SamplingParams(temperature=0, max_tokens=128, skip_special_tokens=False)

baseline = llm.generate(example, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
# Negative scale steers toward the child-oriented end of the style axis.
steering_children = SteeringSpec(
    vectors=[
        VectorSpec(
            source="style-probe.gguf",
            scale=-1.0,
            layers=list(range(28)),
            normalize=True,
            apply=ApplySpec(prompt="all", generation="all"),
        )
    ],
)
output = llm.generate(example, params, steering=steering_children, use_tqdm=False)
print("=====Children=====")
print(output[0].outputs[0].text)

In [ ]:
# Positive scale steers toward the adults-only end of the same axis.
steering_adult = SteeringSpec(
    vectors=[
        VectorSpec(
            source="style-probe.gguf",
            scale=1.5,
            layers=list(range(28)),
            normalize=True,
            apply=ApplySpec(prompt="all", generation="all"),
        )
    ],
)
output = llm.generate(example, params, steering=steering_adult, use_tqdm=False)
print("=====Adult=====")
print(output[0].outputs[0].text)